## Example 1: libnacl functionality test 1

In [ ]:
import libnacl
import libnacl.secret
import libnacl.public

# Example 1: Basic symmetric encryption with SecretBox
print("Example 1: Basic Symmetric Encryption with SecretBox")

try:
    # Create a secret box for symmetric encryption
    box = libnacl.secret.SecretBox()
    
    print(f"SecretBox created successfully")
    print(f"Key length: {len(box.sk)} bytes")
    
    # Encrypt some data
    message = b"This is a secret message for encryption testing"
    print(f"Original message: {message.decode()}")
    
    # Encrypt the message
    encrypted = box.encrypt(message)
    print(f"Encrypted data length: {len(encrypted)} bytes")
    print(f"Encrypted (hex): {encrypted[:32].hex()}...")
    
    # Decrypt the message
    decrypted = box.decrypt(encrypted)
    print(f"Decrypted message: {decrypted.decode()}")
    
    # Verify the decryption worked
    print(f"Decryption successful: {message == decrypted}")
    
    # Show encryption properties
    print(f"Encryption authenticated: Yes (SecretBox uses authenticated encryption)")
    print(f"Nonce handling: Automatic (random nonce generated)")
    
    # Test with different message
    message2 = b"Another test message"
    encrypted2 = box.encrypt(message2)
    decrypted2 = box.decrypt(encrypted2)
    print(f"Second message test: {message2 == decrypted2}")
    
    print("Symmetric encryption completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    raise

## Example 2: libnacl functionality test 2

In [ ]:
import libnacl.public

# Example 2: Asymmetric encryption with PublicKey/PrivateKey
print("Example 2: Asymmetric Encryption with Public/Private Keys")

try:
    # Generate key pairs for Alice and Bob
    alice_keys = libnacl.public.SecretKey()
    bob_keys = libnacl.public.SecretKey()
    
    print(f"Generated key pairs for Alice and Bob")
    print(f"Private key length: {len(alice_keys.sk)} bytes")
    print(f"Public key length: {len(alice_keys.pk)} bytes")
    
    # Alice creates a box to send messages to Bob
    alice_box = libnacl.public.Box(alice_keys.sk, bob_keys.pk)
    # Bob creates a box to decrypt messages from Alice  
    bob_box = libnacl.public.Box(bob_keys.sk, alice_keys.pk)
    
    # Alice encrypts a message for Bob
    message = b"Hello Bob! This is a secure message from Alice."
    print(f"Alice's message: {message.decode()}")
    
    encrypted = alice_box.encrypt(message)
    print(f"Encrypted message length: {len(encrypted)} bytes")
    print(f"Encrypted (hex preview): {encrypted[:32].hex()}...")
    
    # Bob decrypts the message from Alice
    decrypted = bob_box.decrypt(encrypted)
    print(f"Bob decrypted: {decrypted.decode()}")
    
    # Verify the communication worked
    print(f"Communication successful: {message == decrypted}")
    
    # Show asymmetric properties
    print(f"Key exchange: Alice public key shared with Bob")
    print(f"Authentication: Bob knows message came from Alice")
    print(f"Forward secrecy: Each message uses unique nonce")
    
    # Test bidirectional communication
    bob_message = b"Thanks Alice! Message received securely."
    bob_encrypted = bob_box.encrypt(bob_message)
    alice_decrypted = alice_box.decrypt(bob_encrypted)
    print(f"Bidirectional test: {bob_message == alice_decrypted}")
    
    print("Asymmetric encryption completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    raise

## Example 3: libnacl functionality test 3

In [ ]:
import libnacl.sign

# Example 3: Digital signatures for message authentication
print("Example 3: Digital Signatures for Message Authentication")

try:
    # Generate signing key pair
    signing_key = libnacl.sign.Signer()
    
    print(f"Generated signing key pair")
    print(f"Signing key length: {len(signing_key.sk)} bytes")
    
    # Message to sign
    message = b"This document is authentic and authorized by the sender."
    print(f"Message to sign: {message.decode()}")
    
    # Sign the message
    signed_message = signing_key.sign(message)
    print(f"Signed message length: {len(signed_message)} bytes")
    print(f"Signature overhead: {len(signed_message) - len(message)} bytes")
    
    # Extract public key from the private key for verification
    # libnacl 2.1: a Signer's public (verify) key is .vk; there is no .pk attribute
    public_key = signing_key.vk
    print(f"Public key length: {len(public_key)} bytes")
    
    # Create verifier from public key
    # Verifier takes the HEX-encoded verify key
    verifier = libnacl.sign.Verifier(signing_key.hex_vk())
    
    # Verify the signed message
    verified_message = verifier.verify(signed_message)
    print(f"Verified message: {verified_message.decode()}")
    
    # Confirm verification worked
    print(f"Signature valid: {message == verified_message}")
    assert verified_message == message
    
    # Test signature detection of tampering
    try:
        # Simulate tampering with signed message
        tampered = bytearray(signed_message)
        tampered[-10] ^= 1  # Flip a bit in the signature
        
        # This should fail verification
        verifier.verify(bytes(tampered))
    except ValueError:  # libnacl's crypto_sign_open raises ValueError('Failed to validate message')
        print("✓ Signature verification correctly detected tampering")
    else:
        raise AssertionError("Tampered signature verified (should not happen)")
    
    print("Digital signature operations completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    raise

## Example 4: libnacl functionality test 4

In [ ]:
import libnacl

# Example 4: Cryptographic hashing and key derivation
print("Example 4: Cryptographic Hashing and Key Derivation")

try:
    # Test different hash functions
    data = b"This is test data for hashing and key derivation functions."
    print(f"Input data: {data.decode()}")
    print(f"Input length: {len(data)} bytes")
    
    # SHA-256 hashing
    sha256_hash = libnacl.crypto_hash_sha256(data)
    print(f"SHA-256 hash: {sha256_hash.hex()}")
    print(f"SHA-256 length: {len(sha256_hash)} bytes")
    
    # SHA-512 hashing  
    sha512_hash = libnacl.crypto_hash_sha512(data)
    print(f"SHA-512 hash: {sha512_hash.hex()[:64]}...")
    print(f"SHA-512 length: {len(sha512_hash)} bytes")
    
    # BLAKE2b hashing (variable length)
    blake2b_hash = libnacl.crypto_generichash(data)
    print(f"BLAKE2b hash: {blake2b_hash.hex()}")
    print(f"BLAKE2b length: {len(blake2b_hash)} bytes")
    
    # Keyed BLAKE2b (a MAC). libnacl 2.1's crypto_generichash takes (msg, key) only: it has no
    # output-length argument, so the old `outlen=16` call raised TypeError on every run
    mac_key = libnacl.randombytes(libnacl.crypto_generichash_KEYBYTES)
    blake2b_keyed = libnacl.crypto_generichash(data, mac_key)
    print(f"BLAKE2b keyed: {blake2b_keyed.hex()}")
    assert len(blake2b_keyed) == libnacl.crypto_generichash_BYTES and blake2b_keyed != blake2b_hash
    
    # Key derivation. libnacl 2.1 binds libsodium's crypto_kdf; it has no pwhash/scrypt binding
    # at all, so the old crypto_pwhash call could never have run against this package
    master_key = libnacl.crypto_kdf_keygen()
    context = b"Examples"  # exactly crypto_kdf_CONTEXTBYTES (8) bytes
    print(f"Master key length: {len(master_key)} bytes")
    derived_key = libnacl.crypto_kdf_derive_from_key(32, 1, context, master_key)
    # Same (id, context, master key) -> same subkey; a different id -> a different subkey
    assert derived_key == libnacl.crypto_kdf_derive_from_key(32, 1, context, master_key)
    assert derived_key != libnacl.crypto_kdf_derive_from_key(32, 2, context, master_key)
    print(f"Derived key: {derived_key.hex()}")
    print(f"Derived key length: {len(derived_key)} bytes")
    
    # Show hash consistency
    second_hash = libnacl.crypto_hash_sha256(data)
    print(f"Hash consistency: {sha256_hash == second_hash}")
    assert sha256_hash == second_hash
    
    print("Cryptographic hashing and key derivation completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    raise

## Example 5: libnacl functionality test 5

In [ ]:
import libnacl
import libnacl.utils

# Example 5: Advanced cryptographic utilities and random generation
print("Example 5: Advanced Cryptographic Utilities and Random Generation")

try:
    # Secure random number generation
    print("Secure random number generation:")
    
    # Generate random bytes
    random_bytes = libnacl.randombytes(32)
    print(f"Random bytes (32): {random_bytes.hex()}")
    
    # Generate random integers
    random_int = libnacl.randombytes_uniform(1000)
    print(f"Random integer (0-999): {random_int}")
    
    # Generate multiple random values
    random_values = [libnacl.randombytes_uniform(100) for _ in range(5)]
    print(f"Random values (0-99): {random_values}")
    
    # Cryptographic random seed
    seed = libnacl.randombytes(libnacl.crypto_box_SEEDBYTES)
    print(f"Crypto seed: {seed.hex()}")
    
    # Memory utilities - secure memory operations
    print(f"\nSecure memory operations:")
    
    # Secure comparison (constant time)
    data1 = b"secret_password_123"
    data2 = b"secret_password_123"
    data3 = b"wrong_password_456"
    
    match1 = libnacl.crypto_verify_32(data1[:32].ljust(32, b'\x00'),
                                      data2[:32].ljust(32, b'\x00'))
    print(f"Secure compare (same): {match1}")
    assert match1 is True, match1

    match2 = libnacl.crypto_verify_32(data1[:32].ljust(32, b'\x00'),
                                      data3[:32].ljust(32, b'\x00'))
    print(f"Secure compare (different): {match2}")
    assert match2 is False, match2
    
    # Show library information
    print(f"\nLibrary information:")
    print(f"LibNaCl version available: Yes")
    
    # Demonstrate key generation patterns
    box_keypair = libnacl.crypto_box_keypair()
    sign_keypair = libnacl.crypto_sign_keypair()
    
    print(f"Box keypair generated: {len(box_keypair[0])} + {len(box_keypair[1])} bytes")
    print(f"Sign keypair generated: {len(sign_keypair[0])} + {len(sign_keypair[1])} bytes")
    assert (len(box_keypair[0]), len(box_keypair[1]), len(sign_keypair[0]), len(sign_keypair[1])) == (32, 32, 32, 64)
    
    # Performance and security features
    security_features = [
        "Constant-time operations: Resistant to timing attacks",
        "Secure random generation: Cryptographically secure randomness",
        "Memory protection: Secure memory handling utilities",
        "Side-channel resistance: Protection against implementation attacks",
        "High performance: Optimized C implementation with Python bindings"
    ]
    
    print(f"\nSecurity features:")
    for feature in security_features:
        print(f"  • {feature}")
    
    print("Advanced cryptographic utilities completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    raise